# TOFOO Relational Emergence — Colab v1.1

No `git clone`. The notebook fetches the base harness plus the v1.1 prompt-contract fix, validates the instrument, then runs the model. Any interface failure becomes `TEST_INVALID_*`, never a silent research zero.


In [ ]:
!pip -q install -U 'transformers>=4.37' accelerate bitsandbytes requests
import base64, getpass, json, os, subprocess, sys, requests, torch
from pathlib import Path
print('cuda:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Use Runtime → Change runtime type → GPU')
print('gpu:', torch.cuda.get_device_name(0))


## Fetch harness files from private repo

Use a fine-grained GitHub PAT with **Contents: Read** for `nsolland/Valo-Twin`. If a Colab secret named `GITHUB_TOKEN` exists it is used; otherwise this prompts once without echoing the token.


In [ ]:
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
if not token:
    token = getpass.getpass('GitHub PAT (Valo-Twin Contents: Read): ')
if not token: raise RuntimeError('No GitHub token supplied')
headers={'Authorization':f'Bearer {token}','Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
api='https://api.github.com/repos/nsolland/Valo-Twin/contents/experiments/relational-emergence'
for name in ['relational_emergence_v1.py','relational_emergence_v1_1.py']:
    r=requests.get(f'{api}/{name}',params={'ref':'main'},headers=headers,timeout=30)
    if r.status_code != 200: raise RuntimeError(f'GitHub fetch failed for {name}: HTTP {r.status_code}: {r.text[:500]}')
    payload=r.json(); code=base64.b64decode(payload['content']).decode('utf-8')
    Path('/content',name).write_text(code)
    print('fetched:',name,payload['sha'],'lines:',len(code.splitlines()))
token=None; headers=None
harness=Path('/content/relational_emergence_v1_1.py')


## 0A. Deterministic self-check


In [ ]:
subprocess.run([sys.executable,str(harness),'--self-test-only','--out','/content/tofoo_selfcheck_v1_1'],check=True)
selfcheck=json.loads(Path('/content/tofoo_selfcheck_v1_1/manifest.json').read_text())
assert selfcheck['self_check']['prompt_contract']=='PASS',selfcheck
print('SELF CHECK + PROMPT CONTRACT: PASS')


## 0B. End-to-end deterministic mock check

Must return pooled/oracle/relational = 1.0, field precision/recall = 1.0, parse failure = 0.


In [ ]:
mock='/content/tofoo_mock_v1_1'
subprocess.run([sys.executable,str(harness),'--mock','--out',mock],check=True)
m=json.loads(Path(mock,'manifest.json').read_text()); d=m['diagnostics']
assert m['instrument_status']=='VALID_SIGNAL_DISCOVERY_RUN',m
assert d['pooled_raw']==1.0 and d['oracle_field']==1.0 and d['relational']==1.0,d
assert d['field_precision']==1.0 and d['field_recall']==1.0,d
assert d['parse_failure_rate']==0.0,d
print('MOCK ORACLE CHECK: PASS')


## 1. Real Qwen2.5-1.5B run

The real model must pass extraction and known YES/NO controls before experiment scores are accepted.


In [ ]:
out='/content/tofoo_relational_results_v1_1'
p=subprocess.run([sys.executable,str(harness),'--worlds','3','--participants','3','--rounds','2','--model-a','Qwen/Qwen2.5-1.5B-Instruct','--out',out])
manifest=json.loads(Path(out,'manifest.json').read_text())
print(json.dumps(manifest,indent=2))
if p.returncode!=0: raise RuntimeError(f"{manifest.get('instrument_status')}: {manifest.get('error','see diagnostics')}")


## 2. Inspect evidence


In [ ]:
for name in ['summary.json','fields.json','raw_extractions.json']:
    pth=Path(out,name); print('\n###',name); print(json.dumps(json.loads(pth.read_text()),indent=2)[:12000])


### Guardrail

v1.1 is signal discovery only. Research claims still require more worlds/seeds, matched inference budgets, frozen holdouts and independent reproduction.
